# Segmentation Metrics Calculator

Compare a **gold** (expected) segmentation with a **predicted** segmentation.

Format: morphs separated by spaces, roots prefixed with `@`.

Example: `za @hrad a`

In [1]:
# ── Define your words here ──────────────────────────────
gold      = "za @hrad a"
predicted = "za @hra d a"

In [2]:
import re

def parse_boundaries(segmented: str):
    """Extract boundary positions from a segmented word (morphs separated by spaces)."""
    clean = segmented.replace("@", "")
    morphs = re.split(r"\s+", clean.strip())
    raw = "".join(morphs)
    boundaries = []
    pos = 0
    for morph in morphs[:-1]:
        pos += len(morph)
        boundaries.append(pos - 1)
    return raw, boundaries

def parse_root_positions(segmented: str):
    """Extract root morph indices from a segmented word."""
    morphs = re.split(r"\s+", segmented.strip())
    return [i for i, m in enumerate(morphs) if m.startswith("@")]

def levenshtein(a, b):
    """Levenshtein distance between two sequences."""
    n, m = len(a), len(b)
    if n == 0: return m
    if m == 0: return n
    dp = list(range(m + 1))
    for i in range(1, n + 1):
        prev = dp[0]
        dp[0] = i
        for j in range(1, m + 1):
            temp = dp[j]
            if a[i - 1] == b[j - 1]:
                dp[j] = prev
            else:
                dp[j] = 1 + min(prev, dp[j], dp[j - 1])
            prev = temp
    return dp[m]

def levenshtein_score(a, b):
    """1 - (distance / max_len). Returns 1.0 for perfect match."""
    dist = levenshtein(a, b)
    max_len = max(len(a), len(b))
    return 1.0 - dist / max_len if max_len > 0 else 1.0

In [3]:
# ── Segmentation metrics ───────────────────────────────
gold_raw, gold_bounds = parse_boundaries(gold)
pred_raw, pred_bounds = parse_boundaries(predicted)

assert gold_raw == pred_raw, f"Words don't match: '{gold_raw}' vs '{pred_raw}'"

seg_dist  = levenshtein(gold_bounds, pred_bounds)
seg_score = levenshtein_score(gold_bounds, pred_bounds)

print(f"Word:        {gold_raw}")
print(f"Gold:        {gold}")
print(f"Predicted:   {predicted}")
print()
print(f"Gold boundaries:      {gold_bounds}")
print(f"Predicted boundaries: {pred_bounds}")
print()
print(f"Segmentation Levenshtein distance: {seg_dist}")
print(f"Segmentation Levenshtein score:    {seg_score:.4f}")
print(f"Exact match: {seg_dist == 0}")

Word:        zahrada
Gold:        za @hrad a
Predicted:   za @hra d a

Gold boundaries:      [1, 5]
Predicted boundaries: [1, 4, 5]

Segmentation Levenshtein distance: 1
Segmentation Levenshtein score:    0.6667
Exact match: False


In [4]:
# ── Root identification metrics ─────────────────────────
gold_roots = parse_root_positions(gold)
pred_roots = parse_root_positions(predicted)

root_dist  = levenshtein(gold_roots, pred_roots)
root_score = levenshtein_score(gold_roots, pred_roots)

print(f"Gold root positions:      {gold_roots}")
print(f"Predicted root positions: {pred_roots}")
print()
print(f"Root Levenshtein distance: {root_dist}")
print(f"Root Levenshtein score:    {root_score:.4f}")
print(f"Exact match: {root_dist == 0}")

Gold root positions:      [1]
Predicted root positions: [1]

Root Levenshtein distance: 0
Root Levenshtein score:    1.0000
Exact match: True
